# Homework 7 - QP3: Bernstein-Vazrani Algorithm (110 pts)

Both Deutsch-Josza (DJ) algorithm and Bernstein-Vazirani (BV) algorithm demonstrate that quantum computers can significantly accelerate the function-query process compared to classical computers by utilizing the linear superposition of quantum states. Both algorithms utilize the following function oracle.

![fig1](/Users/yuyaohuang/Github/EE520_QC_and_device/HW7/QP-3_fig1.png)

The x-register is a query register, where the function oracle $U_f$ allows a 'phase-type' query where the x-register picks up a phase $(-1)^{f(x)} = e^{i\pi f(x)}$ that depends on the function value f(x).  This phase factor is shown in magenta color in the state-(1) after the oracle. Upon a subsequent Hadamard transform, the x-register becomes state-(2). In this state-(2) with a double summation, the x-summation gives a clue about the solution to the $\text{DJ}$ problem and to the $\text{BV}$ problem. For example, in the $\text{DJ}$ problem, we examine the $|z\rangle = |0\rangle$ state of the x-register, and the x-summation (in state-(2)) will yield a **zero** for a **balanced** function [$f(x)$ *s 0 for half the x and 1 for the other half of x*], and $\pm 2^n$ for a **constant** function $f(x)$ [$+ 2^n$ *if* $f(x)$ *is 0 for all x*; $- 2^n$ *if* $f(x)$ *is 1 for all x*].

Here, the quantum speedup is made possible by the uniform superposition state of the x-register at the input of the oracle. This superposition leads to a one-shot or parallel (simultaneous) calculation of the function value $f(x)$ for all possible ($2^n$ *different*) x argument values.

The quantum algorithms involving a function oracle $U_f$ usually resort to this kind of phase-type query as seen in state-(1), with ($-1^{f(x)}$). This kind of phase-type oracle is used with the Quantum Search algorithm to solve problems.

[<u>Comment:</u> *Note the simple math in **the Uniform Superposition state** where the **tensor product of n terms $|+\rangle ^{\otimes n}$** on the LHS is equal to a **summation of $2^n$ terms** on the RHS.* A similar observation works for the exponential speedup of the Quantum Fourier Transform over the classical Fourier Transform, as we will explore later.]

<u>QP-3 Assignment</u> Write a circuit for the Bernstein-Vazrani algorithm with the x-register having any number of qubits that is greater than 5 qubits. A single-qubit y-register is put in the $|-\rangle$ state before the Oracle.   Make a randomly generated oracle circuit (a code segment is given below). In the $\text{BV}$ algorithm, every qubit exists in a *separable* state throughout circuit and therefore, you can keep track of the individual qubit state throughout.  You should measure only the x-register output. 

Report:

(i) Your quantum circuit.  On this circuit, write down the qubit state after each gate for every qubit throughout the circuit. (30 pts)

(ii) Simulation result histogram.  Discuss your result. (15 pts)

(iii) IBM System computation result histogram. Discuss your result. (15 pts)

(iv) Use the magenta-colored x-summation in the **statevector-(2) of the x-register** (in the previous page) and **show** why this x-summation yields a **nonzero value (amplitude)** only for the $|z\rangle = |s\rangle$ state and a **zero value** for all other states, i.e., $|z\rangle \ne |s\rangle$ of the x-register. What is the value of the magenta-colored x-summation for the $|z\rangle$ state with z = your s? What is its value if z $\ne$ s? <u>Hint:</u> *Do the x-summation by breaking it into a binary summation ($x_i = 0,1,$ and $x = x_{n-1} \cdots X_1 X_0$) instead of doing the integer summation over $x=0,1,2,\cdots,2^n-1$*.

(v) **Show:** (a) $\mathrm{CNOT} |x\rangle |-\rangle = (-1)^x |x\rangle |-\rangle$; and find: (b) $\mathrm{CNOT} |+\rangle |-\rangle = (\text{     })$ and $\mathrm{CNOT} |-\rangle |-\rangle = (\text{     })$. For the function oracle $U_f$, **show:** (c) $U_f |x\rangle |-\rangle = (-1)^{f(x)} |x\rangle |-\rangle$; and **find**, for a single-bit x-register. (15 pts)

(vi) In the previous page, with an n-bit x-register: (a) **Derive** the **statevector-(1)** after $U_f$; (b) **Derive** the **statevector-(1)** after $H \otimes n$.

<u>Comments:</u>
The following code segment generates a function Oracle for a randomly generated s-string (you can see what s-string you made by *print(s, s_str)*). Note that 'nqubits' is the number of qubits in the x-register.

```Python
## Generate the bit string "s" 
s=np.random.randint(2**nqubits)  #integer in 0,..,2^nqubits - 1 
s_str=format(s,'0'+str(nqubits)+'b') #convert int s to binary string s_str 
# print(s, s_str) 
## Build an Oracle with CNOT's from bit-'s'(if s=1) in x-register to y-register in |->  
for i in range(nqubits): 
    if(s&(1<<i)): #if ith bit of 's' is not 0 
        circuit.cx(i,nqubits)  #CNOT from a qubit in x-register to Target(y-register) 
circuit.barrier()
```

In [9]:
"""Part (i) — BV Quantum Circuit and Qubit State Tracking"""

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# ── Parameters ────────────────────────────────────────────────────────────────
nqubits = 6          # x-register size: n = 6 (> 5 as required)

# ── Randomly generate the secret bitstring s ─────────────────────────────────
np.random.seed(520)  # fixed seed for reproducibility; remove for a random s
s     = np.random.randint(2**nqubits)
s_str = format(s, '0' + str(nqubits) + 'b')   # MSB first: x5 x4 x3 x2 x1 x0
print(f"Secret string s = {s_str}  (integer: {s})")
print(f"  bit order (index): x5 x4 x3 x2 x1 x0 = {s_str}")

# ── Build the BV Circuit ──────────────────────────────────────────────────────
qr_x = QuantumRegister(nqubits, 'x')   # x-register: query qubits
qr_y = QuantumRegister(1,       'y')   # y-register: ancilla
cr   = ClassicalRegister(nqubits, 'c') # classical bits for measurement
circuit = QuantumCircuit(qr_x, qr_y, cr)

# Step 1: Initialize y-register to |1⟩  via X gate
# State: x_i = |0⟩,  y = |1⟩
circuit.x(nqubits)
circuit.barrier()

# Step 2: Hadamard on ALL qubits
# x_i: |0⟩ → |+⟩ = (|0⟩+|1⟩)/√2   for all i
# y:   |1⟩ → |−⟩ = (|0⟩−|1⟩)/√2
circuit.h(range(nqubits))   # x-register
circuit.h(nqubits)           # y-register
circuit.barrier()

# Step 3: Oracle U_f — CNOT from x_i to y whenever s_i = 1
# Phase kickback: CNOT|x_i⟩|−⟩ = (−1)^{x_i}|x_i⟩|−⟩
# Result: x_i state after oracle:
#   if s_i = 0 → no CNOT → x_i stays |+⟩ = (|0⟩+|1⟩)/√2
#   if s_i = 1 → CNOT  → x_i becomes |−⟩ = (|0⟩−|1⟩)/√2
# y stays |−⟩ throughout (eigenvector of CNOT target)
for i in range(nqubits):
    if s & (1 << i):        # if s_i = 1
        circuit.cx(i, nqubits)
circuit.barrier()

# Step 4: Hadamard on x-register only
# H|+⟩ = |0⟩  (when s_i = 0)
# H|−⟩ = |1⟩  (when s_i = 1)
# → x_i collapses to |s_i⟩ deterministically
circuit.h(range(nqubits))
circuit.barrier()

# Step 5: Measure x-register
circuit.measure(qr_x, cr)

# ── Print Per-Qubit State Tracking ───────────────────────────────────────────
print("\n" + "="*72)
print("  QUBIT STATE TRACKING THROUGH THE BV CIRCUIT")
print("="*72)
print(f"{'Qubit':<8} {'Init':>6} {'After X(y)':>12} {'After H':>10}"
      f" {'After Oracle':>16} {'After H(x)':>12}")
print("-"*72)
for i in range(nqubits):
    s_i    = (s >> i) & 1
    oracle = "|+⟩" if s_i == 0 else "|−⟩"
    final  = f"|{s_i}⟩"
    print(f"x{i:<7} {'|0⟩':>6} {'|0⟩':>12} {'|+⟩':>10} {oracle:>16} {final:>12}")
print(f"{'y':<8} {'|0⟩':>6} {'|1⟩':>12} {'|−⟩':>10} {'|−⟩':>16} {'|−⟩':>12}")
print("-"*72)

# ── Overall Register State at Each Step ──────────────────────────────────────
print("\nOverall x-register ⊗ y-register state at each step:")
print(f"  Step 0 (init)      : |0⟩^⊗{nqubits} ⊗ |0⟩")
print(f"  Step 1 (X on y)    : |0⟩^⊗{nqubits} ⊗ |1⟩")
print(f"  Step 2 (H on all)  : |+⟩^⊗{nqubits} ⊗ |−⟩  =  (1/√2^{nqubits}) Σ_x |x⟩ ⊗ |−⟩")
oracle_state = ' ⊗ '.join(
    f"({'|+⟩' if (s >> i) & 1 == 0 else '|−⟩'})" for i in range(nqubits - 1, -1, -1)
)
print(f"  Step 3 (Oracle)    : {oracle_state} ⊗ |−⟩")
print(f"                     = (1/√2^{nqubits}) Σ_x (-1)^{{f(x)}} |x⟩ ⊗ |−⟩")
print(f"  Step 4 (H on x)    : |{s_str}⟩ ⊗ |−⟩  =  |s⟩ ⊗ |−⟩")
print(f"\n→ Measurement collapses x-register to |s⟩ = |{s_str}⟩  (integer {s}) with certainty.")

# ── Draw the Circuit ──────────────────────────────────────────────────────────
fig = circuit.draw(output='mpl', style='iqp', fold=25)
fig.suptitle(f"BV Circuit  —  n={nqubits} qubits,  secret s = {s_str}", fontsize=12)
plt.tight_layout()
plt.show()


Secret string s = 101100  (integer: 44)
  bit order (index): x5 x4 x3 x2 x1 x0 = 101100

  QUBIT STATE TRACKING THROUGH THE BV CIRCUIT
Qubit      Init   After X(y)    After H     After Oracle   After H(x)
------------------------------------------------------------------------
x0          |0⟩          |0⟩        |+⟩              |+⟩          |0⟩
x1          |0⟩          |0⟩        |+⟩              |+⟩          |0⟩
x2          |0⟩          |0⟩        |+⟩              |−⟩          |1⟩
x3          |0⟩          |0⟩        |+⟩              |−⟩          |1⟩
x4          |0⟩          |0⟩        |+⟩              |+⟩          |0⟩
x5          |0⟩          |0⟩        |+⟩              |−⟩          |1⟩
y           |0⟩          |1⟩        |−⟩              |−⟩          |−⟩
------------------------------------------------------------------------

Overall x-register ⊗ y-register state at each step:
  Step 0 (init)      : |0⟩^⊗6 ⊗ |0⟩
  Step 1 (X on y)    : |0⟩^⊗6 ⊗ |1⟩
  Step 2 (H on all)  : |+⟩^⊗6 ⊗ |−

<Figure size 640x480 with 0 Axes>

In [13]:
"""Part (ii) — Simulation Result and Discussion"""

shots = 1024

# ── Run simulation with AerSimulator ─────────────────────────────────────────
simulator = AerSimulator()
compiled  = transpile(circuit, simulator)
job       = simulator.run(compiled, shots=shots)
counts    = job.result().get_counts()

# Qiskit counts key: string is printed MSB (highest qubit index) first,
# which matches s_str directly (s_str is also MSB = x5 first).
print(f"Secret s_str (x5→x0 order) : {s_str}")
print(f"All counts                 : {counts}")
print(f"Count for s                : {counts.get(s_str, 0)} / {shots}"
      f"  ({100 * counts.get(s_str, 0) / shots:.1f}%)")

# ── Plot histogram ────────────────────────────────────────────────────────────
plot_histogram(counts,
               title=f"BV Simulation  (n={nqubits}, s={s_str}, shots={shots})",
               color='steelblue', figsize=(6, 4))
plt.tight_layout()
plt.show()


Secret s_str (x5→x0 order) : 101100
All counts                 : {'101100': 1024}
Count for s                : 1024 / 1024  (100.0%)


<Figure size 640x480 with 0 Axes>

### Part (ii) — Discussion

The simulation histogram shows **a single bar at $s = 101100$** with all 1024 shots (100% probability), confirming that the BV algorithm recovers the hidden bitstring **deterministically in a single oracle query**.

**Why the result is deterministic:**

In the BV algorithm every qubit remains in a **separable (product) state** throughout the circuit — no entanglement between x-qubits is ever created. This means each qubit $x_i$ evolves independently:

| Step | $x_i$ (if $s_i = 0$) | $x_i$ (if $s_i = 1$) |
|:---|:---|:---|
| After $H$ | $\|+\rangle$ | $\|+\rangle$ |
| After Oracle (phase kickback) | $\|+\rangle$ (no CNOT) | $\|-\rangle$ (phase $-1$ acquired) |
| After final $H$ | $H\|+\rangle = \|0\rangle$ | $H\|-\rangle = \|1\rangle$ |

The final Hadamard uniquely inverts the phase encoding: $\|+\rangle \to \|0\rangle = \|s_i\!=\!0\rangle$ and $\|-\rangle \to \|1\rangle = \|s_i\!=\!1\rangle$. Every qubit collapses to exactly $\|s_i\rangle$ — no probability is wasted on wrong outcomes.

**Quantum speedup:** A classical algorithm requires $n = 6$ oracle queries (one per bit of $s$), while the BV quantum algorithm requires only **1 query** regardless of $n$.


In [12]:
"""Part (iii) — IBM Quantum System Result and Discussion
NOTE: This cell requires a personal IBM Quantum API key in ../apikey.json.
It was executed on 2026-04-17 on backend ibm_fez (Job ID: d7grr0s93s0c738rva2g).
The histogram and counts are preserved from that run.
Re-running requires valid credentials and will submit a new job.
"""

import json
from pathlib import Path
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# ── Load API key from file (never hardcode credentials) ──────────────────────
key_path = Path("../apikey.json")
with open(key_path) as f:
    api_key = json.load(f)["apikey"]

# ── Connect to IBM Quantum and pick the least-busy real backend ───────────────
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=api_key)
backend = service.least_busy(operational=True, simulator=False,
                              min_num_qubits=nqubits + 1)
print(f"Using backend : {backend.name}")

# ── Transpile for the selected backend ───────────────────────────────────────
pm       = generate_preset_pass_manager(target=backend.target, optimization_level=1)
compiled = pm.run(circuit)
print(f"Circuit depth after transpilation: {compiled.depth()}")

# ── Submit job and wait ───────────────────────────────────────────────────────
sampler = Sampler(mode=backend)
job_ibm = sampler.run([compiled], shots=shots)
print(f"Job ID : {job_ibm.job_id()}  (status: {job_ibm.status()})")
print("Waiting for IBM job to complete...")

# ── Retrieve and display results ──────────────────────────────────────────────
result_ibm = job_ibm.result()
pub_result = result_ibm[0]
counts_ibm = pub_result.data.c.get_counts()   # classical register 'c'

print(f"\nIBM real-system counts (shots={shots}) :", counts_ibm)
top_result = max(counts_ibm, key=counts_ibm.get)
correct    = counts_ibm.get(s_str, 0)
print(f"Most frequent bitstring : {top_result}")
print(f"Secret s (x5→x0 order)  : {s_str}")
print(f"Correct (s) count       : {correct} / {shots}  ({100*correct/shots:.1f}%)")

# ── Plot histogram ────────────────────────────────────────────────────────────
plot_histogram(counts_ibm,
               title=f"IBM Real System ({backend.name})  —  BV n={nqubits}, s={s_str}",
               color='darkorange', figsize=(8, 4))
plt.tight_layout()
plt.show()


qiskit_runtime_service._discover_account:WARNING:2026-04-17 00:56:28,430: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-04-17 00:56:30,685: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-17 00:56:31,403: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-04-17 00:56:33,815: Using instance: open-instance, plan: open


Using backend : ibm_fez
Circuit depth after transpilation: 17
Job ID : d7grr0s93s0c738rva2g  (status: QUEUED)
Waiting for IBM job to complete...

IBM real-system counts (shots=1024) : {'101100': 851, '111100': 38, '101110': 37, '100100': 32, '101000': 16, '101101': 23, '000100': 4, '111000': 1, '001100': 14, '000000': 2, '001000': 1, '001110': 2, '100000': 1, '111101': 1, '101001': 1}
Most frequent bitstring : 101100
Secret s (x5→x0 order)  : 101100
Correct (s) count       : 851 / 1024  (83.1%)


<Figure size 640x480 with 0 Axes>

### Part (iii) — Discussion

The IBM real-system result on **ibm_fez** (circuit depth = 17 after transpilation, Job ID: `d7grr0s93s0c738rva2g`) is:

| Bitstring | Count | Probability | Bits flipped from $s$ |
|:---|:---:|:---:|:---:|
| **101100** (correct $s$) | **851** | **83.1%** | — |
| 111100 | 38 | 3.7% | $x_4$: $0\to1$ |
| 101110 | 37 | 3.6% | $x_1$: $0\to1$ |
| 100100 | 32 | 3.1% | $x_3$: $1\to0$ |
| 101101 | 23 | 2.2% | $x_0$: $0\to1$ |
| 101000 | 16 | 1.6% | $x_2$: $1\to0$ |
| 001100 | 14 | 1.4% | $x_5$: $1\to0$ |
| others | 13 | 1.3% | ≥2 bits |

$s = \mathtt{101100}$ remains the **dominant peak** with 83.1% of shots, correctly identifying the hidden string. Compared to the ideal simulation (100%), about **16.9% of shots yielded wrong bitstrings**.

**Sources of error on real hardware:**

1. **Readout (measurement) errors**: The most striking pattern is that every major wrong bitstring differs from $s$ by **exactly 1 bit** — one per qubit index $x_0$ through $x_5$. This is the hallmark of independent single-qubit readout flips, which are the dominant error source on current superconducting processors.

2. **CNOT (two-qubit gate) errors**: The transpiled circuit has depth 17, requiring multiple CNOT gates beyond the 3 oracle CNOTs (transpilation adds SWAP/bridge gates). Each CNOT on ibm_fez carries a typical error rate of ~0.1–0.5%, and these compound over the circuit depth.

3. **Decoherence ($T_1$/$T_2$)**: The circuit depth of 17 is well within ibm_fez's coherence limits, so decoherence contributes only minor additional error compared to gate and readout errors.

**Conclusion:** Despite hardware noise, the BV algorithm clearly identifies $s = \mathtt{101100}$ in a single oracle query with 83.1% fidelity. The strong single-bit-flip pattern in the error distribution confirms that readout errors are the primary noise source on ibm_fez for this circuit.


## Part (iv) — Why the x-Summation is Nonzero Only for $|z\rangle = |s\rangle$

In statevector-(2), the amplitude of basis state $|z\rangle$ in the x-register is proportional to the **x-summation**:

$$A_z = \sum_{x=0}^{2^n-1} (-1)^{f(x)\,+\,z\cdot x}$$

For the BV problem $f(x) = s\cdot x \pmod{2}$, so $(-1)^{f(x)} = (-1)^{s\cdot x}$.  Substituting:

$$A_z = \sum_{x=0}^{2^n-1} (-1)^{(s\oplus z)\cdot x} = \sum_{x=0}^{2^n-1} (-1)^{\,\sum_{i=0}^{n-1}(s_i+z_i)\,x_i}$$

### Binary Decomposition

Write $x = x_{n-1}\cdots x_1 x_0$ with each $x_i\in\{0,1\}$.  The sum over all $2^n$ values of $x$ factors into **independent single-bit sums**:

$$A_z = \prod_{i=0}^{n-1}\underbrace{\left[\sum_{x_i=0}^{1}(-1)^{(s_i+z_i)\,x_i}\right]}_{F_i}$$

Evaluate each factor $F_i$:

| Condition | $F_i$ |
|:---|:---|
| $s_i = z_i$ | $(-1)^0+(-1)^0 = 1+1 = 2$ |
| $s_i \neq z_i$ | $(-1)^0+(-1)^1 = 1-1 = 0$ |

Therefore:

$$A_z = \prod_{i=0}^{n-1} F_i = \begin{cases} 2^n & \text{if } z = s \quad (\text{every bit matches} \Rightarrow \text{every } F_i = 2) \\ 0 & \text{if } z \neq s \quad (\text{at least one bit differs} \Rightarrow \text{some } F_i = 0) \end{cases}$$

### Full Amplitude in Statevector-(2)

The amplitude of $|z\rangle$ is $\dfrac{A_z}{2^n}$:

$$\langle z|\psi_2\rangle = \frac{A_z}{2^n} = \begin{cases} 1 & z = s \\ 0 & z \neq s \end{cases}$$

**For $z = s$ (your secret $s$):** the x-summation equals $A_s = 2^n = 2^6 = 64$.

**For any $z \neq s$:** the x-summation equals $A_z = 0$.

This proves the BV algorithm recovers $s$ **with certainty in a single oracle query**.


## Part (v) — CNOT and Oracle Phase-Kickback Derivations

### (a) Show: $\mathrm{CNOT}\,|x\rangle|-\rangle = (-1)^x\,|x\rangle|-\rangle$

Recall $|-\rangle = \dfrac{1}{\sqrt{2}}\bigl(|0\rangle - |1\rangle\bigr)$.  The CNOT gate flips the target qubit when the control is $|1\rangle$.

**Case $x = 0$:** control is $|0\rangle$, so CNOT leaves the target unchanged:
$$\mathrm{CNOT}\,|0\rangle|-\rangle = |0\rangle\cdot\frac{|0\rangle-|1\rangle}{\sqrt{2}} = (-1)^0\,|0\rangle|-\rangle \checkmark$$

**Case $x = 1$:** control is $|1\rangle$, so CNOT flips target ($|0\rangle\leftrightarrow|1\rangle$):
$$\mathrm{CNOT}\,|1\rangle|-\rangle = |1\rangle\cdot\frac{|1\rangle-|0\rangle}{\sqrt{2}} = -|1\rangle\cdot\frac{|0\rangle-|1\rangle}{\sqrt{2}} = (-1)^1\,|1\rangle|-\rangle \checkmark$$

In both cases: $\boxed{\mathrm{CNOT}\,|x\rangle|-\rangle = (-1)^x\,|x\rangle|-\rangle}$  (**phase kickback**)  $\square$

---

### (b) Find: $\mathrm{CNOT}\,|+\rangle|-\rangle$ and $\mathrm{CNOT}\,|-\rangle|-\rangle$

Using result (a) by linearity of quantum gates:

$$\mathrm{CNOT}\,|+\rangle|-\rangle = \mathrm{CNOT}\cdot\frac{|0\rangle+|1\rangle}{\sqrt{2}}\,|-\rangle = \frac{(-1)^0\,|0\rangle+(-1)^1\,|1\rangle}{\sqrt{2}}\,|-\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}\,|-\rangle = \boxed{|-\rangle|-\rangle}$$

$$\mathrm{CNOT}\,|-\rangle|-\rangle = \mathrm{CNOT}\cdot\frac{|0\rangle-|1\rangle}{\sqrt{2}}\,|-\rangle = \frac{(-1)^0\,|0\rangle-(-1)^1\,|1\rangle}{\sqrt{2}}\,|-\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}\,|-\rangle = \boxed{|+\rangle|-\rangle}$$

---

### (c) Show: $U_f\,|x\rangle|-\rangle = (-1)^{f(x)}\,|x\rangle|-\rangle$

For the BV oracle, $f(x) = s\cdot x \bmod 2 = \bigoplus_{i=0}^{n-1} s_i x_i$.  The oracle applies $\mathrm{CNOT}_{x_i\to y}$ for each $i$ with $s_i=1$, in any order (they commute since $y$ is the common target).

Since $|x\rangle$ is a fixed computational basis state, each applied CNOT contributes an independent phase $(-1)^{x_i}$ from result (a):

$$U_f\,|x\rangle|-\rangle = \prod_{i:\,s_i=1}(-1)^{x_i}\cdot|x\rangle|-\rangle = (-1)^{\sum_i s_i x_i}\,|x\rangle|-\rangle = \boxed{(-1)^{f(x)}\,|x\rangle|-\rangle} \quad\square$$

**For a single-bit x-register** ($n=1$, $f(x_0) = s_0 x_0$):

| $s_0$ | $x_0$ | $f(x_0)$ | $U_f\,|x_0\rangle|-\rangle$ |
|:---:|:---:|:---:|:---|
| 0 | 0 | 0 | $+\,|0\rangle\,|-\rangle$ |
| 0 | 1 | 0 | $+\,|1\rangle\,|-\rangle$ |
| 1 | 0 | 0 | $+\,|0\rangle\,|-\rangle$ |
| 1 | 1 | 1 | $-\,|1\rangle\,|-\rangle$ |

When $s_0=0$: $U_f$ is the identity on $|x_0\rangle|-\rangle$.  When $s_0=1$: $U_f$ applies $(-1)^{x_0}$ phase, acting as a **phase gate** on $|x_0\rangle$ while leaving $|-\rangle$ unchanged.

---

### (d) Find $U_f\,|+\rangle|-\rangle$ for a single-bit x-register

Using $|+\rangle = \dfrac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and applying $U_f$ by linearity (result (c)):

$$U_f\,|+\rangle|-\rangle = \frac{1}{\sqrt{2}}\bigl((-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle\bigr)|-\rangle$$

**Case $s_0 = 0$** ($f(0) = 0,\; f(1) = 0$):
$$U_f\,|+\rangle|-\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)|-\rangle = \boxed{|+\rangle\,|-\rangle}$$

**Case $s_0 = 1$** ($f(0) = 0,\; f(1) = 1$):
$$U_f\,|+\rangle|-\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle - |1\rangle\bigr)|-\rangle = \boxed{|-\rangle\,|-\rangle}$$

**Summary:**
$$U_f\,|+\rangle|-\rangle = \begin{cases}|+\rangle\,|-\rangle & s_0 = 0\\|-\rangle\,|-\rangle & s_0 = 1\end{cases}$$

This result directly illustrates the phase-kickback mechanism: the hidden bit $s_0$ is encoded into the phase of the x-qubit ($|+\rangle$ vs $|-\rangle$), which a final Hadamard then converts to $|0\rangle$ or $|1\rangle$ deterministically.


## Part (vi) — Derivation of Statevectors (1) and (2)

### (a) Statevector-(1): State after the Oracle $U_f$

**Initial state** before any gate: $|0\rangle^{\otimes n}|0\rangle$

**After $X$ on y and $H^{\otimes(n+1)}$ on all qubits:**

$$|\psi_0\rangle = H^{\otimes n}|0\rangle^{\otimes n}\otimes H|1\rangle = \left(\frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle\right)\otimes|-\rangle = \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle\,|-\rangle$$

where $H|1\rangle = |-\rangle = \dfrac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$.

**Applying $U_f$** using result (v)(c) term-by-term (linearity):

$$\boxed{|\psi_1\rangle = U_f|\psi_0\rangle = \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}(-1)^{f(x)}\,|x\rangle\,|-\rangle}$$

Each basis state $|x\rangle$ picks up a phase $(-1)^{f(x)}$ while the y-register remains $|-\rangle$ throughout.  This is the **phase-type query**: the function value $f(x)$ is encoded in the phase of $|x\rangle$, not in a separate output register.

---

### (b) Statevector-(2): State after the final $H^{\otimes n}$ on the x-register

Apply $H^{\otimes n}$ to the x-register using the identity:

$$H^{\otimes n}|x\rangle = \frac{1}{\sqrt{2^n}}\sum_{z=0}^{2^n-1}(-1)^{x\cdot z}|z\rangle, \qquad x\cdot z = \sum_{i=0}^{n-1}x_i z_i \pmod{2}$$

Substituting into $|\psi_1\rangle$:

$$|\psi_2\rangle = H^{\otimes n}|\psi_1\rangle = \frac{1}{\sqrt{2^n}}\sum_{x}(-1)^{f(x)}\left[\frac{1}{\sqrt{2^n}}\sum_{z}(-1)^{x\cdot z}|z\rangle\right]|-\rangle$$

Collecting terms by $|z\rangle$:

$$\boxed{|\psi_2\rangle = \sum_{z=0}^{2^n-1}\underbrace{\left(\frac{1}{2^n}\sum_{x=0}^{2^n-1}(-1)^{f(x)+x\cdot z}\right)}_{\text{amplitude of }|z\rangle}|z\rangle\,|-\rangle}$$

From Part (iv), the amplitude (x-summation divided by $2^n$) equals $\delta_{z,s}$:

$$|\psi_2\rangle = \sum_{z}\delta_{z,s}\,|z\rangle\,|-\rangle = |s\rangle\,|-\rangle$$

**Measuring the x-register always yields $s$ with probability 1** — the hidden bitstring is recovered in a **single oracle query**.
